## Caso não tenha as libs instaladas no Kernel

In [1]:
# %pip install plotly pandas scikit-learn opencv-python
# %pip install --upgrade nbformat

## Import das libs

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
import json
from pathlib import Path

from IPython.display import Markdown, display
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

### Analise dos logs de voo em memmap

In [3]:
lista_dfs = []
caminhos_ficheiros = []


def listar_logs_voo_memmap():
    candidatos = sorted(Path("../logs").glob("voo_teste_*/manifest.json"))
    if not candidatos:
        candidatos = sorted(Path("logs").glob("voo_teste_*/manifest.json"))
    return candidatos

def carregar_log_voo_memmap(manifest_path):
    """Carrega intervalos de voo e reconstrui trajetoria e tempo relativos."""

    manifest_path = Path(manifest_path)

    try:
        texto = manifest_path.read_text(encoding="utf-8").strip()

        if not texto:
            print(f"Ignorando manifest vazio: {manifest_path}")
            return pd.DataFrame()

        manifest = json.loads(texto)

    except JSONDecodeError as e:
        print(f"Ignorando manifest invÃ¡lido: {manifest_path}")
        print(f"Erro: {e}")
        return pd.DataFrame()

    n = int(manifest.get("num_samples", 0))
    if n <= 0:
        return pd.DataFrame()

    intervalos_path = manifest_path.parent / manifest["arrays"]["flight_intervals"]

    if not intervalos_path.exists():
        print(f"Ignorando run sem memmap: {intervalos_path}")
        return pd.DataFrame()

    intervalos = np.load(intervalos_path, mmap_mode="r")[:n]

    df = pd.DataFrame(intervalos)
    df["run_id"] = manifest_path.parent.name
    df["manifest_path"] = str(manifest_path)

    df["tempo_s"] = df["dt_s"].fillna(0).cumsum()
    df["x_rel"] = df["delta_x_m"].fillna(0).cumsum()
    df["y_rel"] = df["delta_y_m"].fillna(0).cumsum()
    df["z_rel"] = df["delta_z_m"].fillna(0).cumsum()
    df["altitude_rel"] = -df["z_rel"]

    return df

for manifest_path in listar_logs_voo_memmap():
    df_temp = carregar_log_voo_memmap(manifest_path)
    if not df_temp.empty:
        caminhos_ficheiros.append(manifest_path)
        lista_dfs.append(df_temp)

if not lista_dfs:
    display(Markdown("Nenhum log novo em memmap encontrado em `logs/voo_teste_*/manifest.json`."))

### Deslocamento acumulado relativo

In [4]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    timestamp = caminho.parent.name.replace("voo_teste_", "")
    fig_3d = go.Figure()
    fig_3d.add_trace(go.Scatter3d(x=df["x_rel"], y=df["y_rel"], z=df["altitude_rel"], mode="lines", line=dict(color="royalblue", width=4), name="Deslocamento acumulado"))
    fig_3d.add_trace(go.Scatter3d(x=[0.0], y=[0.0], z=[0.0], mode="markers", marker=dict(color="green", size=6), name="Origem relativa"))
    fig_3d.add_trace(go.Scatter3d(x=[df["x_rel"].iloc[-1]], y=[df["y_rel"].iloc[-1]], z=[df["altitude_rel"].iloc[-1]], mode="markers", marker=dict(color="red", size=6, symbol="x"), name="Fim relativo"))
    fig_3d.update_layout(title=f"Deslocamento acumulado por deltas - Run: {timestamp}", scene=dict(xaxis_title="Delta X acumulado (m)", yaxis_title="Delta Y acumulado (m)", zaxis_title="Delta altitude acumulada (m)", camera=dict(eye=dict(x=1.5, y=1.5, z=0.5))), legend=dict(x=0, y=1), margin=dict(l=0, r=0, b=0, t=40))
    fig_3d.show()

### Variacao das velocidades angulares

In [5]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    timestamp = caminho.parent.name.replace("voo_teste_", "")
    fig_2d = go.Figure()
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_roll_speed_rad_s"], mode="lines", name="Delta roll speed", opacity=0.7))
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_pitch_speed_rad_s"], mode="lines", name="Delta pitch speed", opacity=0.7))
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_yaw_speed_rad_s"], mode="lines", name="Delta yaw speed", opacity=0.7))
    fig_2d.update_layout(title=f"Variacao das velocidades angulares - Run: {timestamp}", xaxis_title="Tempo acumulado por intervalos (s)", yaxis_title="Delta velocidade angular (rad/s)", template="plotly_white", hovermode="x unified")
    fig_2d.show()

### Analise dos intervalos depth/flow em memmap

Esta analise usa as novas runs em `datasets/depth_ground_truth/run_*/manifest.json`. Cada amostra representa a variacao entre duas atualizacoes consecutivas da logica de proximidade visual por optical flow, a mesma logica que gera as flechas desenhadas na deteccao de obstaculos.

O foco deixa de ser o estado absoluto do drone em um frame isolado e passa a ser o deslocamento sincronizado do intervalo: deltas de posicao, atitude, IMU, comandos reativos, flow e depth ground truth. O objetivo e deixar os dados menos dependentes da origem da simulacao e mais genericos para treino e analise.

In [6]:
def localizar_raiz_projeto_memmap(nome_dataset="datasets"):
    """Localiza a raiz do projeto a partir do diretorio do notebook."""

    candidatos = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidato in candidatos:
        if (candidato / nome_dataset / "depth_ground_truth").exists():
            return candidato
    return Path.cwd()


def listar_runs_depth_memmap(base_dir):
    depth_dir = base_dir / "datasets" / "depth_ground_truth"
    return sorted(path.parent for path in depth_dir.glob("run_*/manifest.json"))


def carregar_manifesto_memmap(run_dir):
    with open(run_dir / "manifest.json", encoding="utf-8") as fp:
        return json.load(fp)


def abrir_array_memmap(run_dir, manifesto, chave):
    return np.load(run_dir / manifesto["arrays"][chave], mmap_mode="r")


def indices_intervalos_sincronizados(intervalos, max_age_s=0.08, max_interval_s=0.5):
    """Seleciona intervalos RGB/depth com tempos validos e sincronizados."""

    if intervalos.empty:
        return np.array([], dtype=int)
    dt_rgb = pd.to_numeric(intervalos["dt_s"], errors="coerce")
    dt_depth = pd.to_numeric(intervalos["depth_dt_s"], errors="coerce")
    depth_age = pd.to_numeric(intervalos["depth_age_s"], errors="coerce")
    mascara = (
        dt_rgb.notna() & dt_rgb.gt(0.0) & dt_rgb.le(max_interval_s)
        & dt_depth.notna() & dt_depth.gt(0.0) & dt_depth.le(max_interval_s)
        & depth_age.notna() & depth_age.le(max_age_s)
    )
    return np.flatnonzero(mascara.to_numpy())


def carregar_run_depth_memmap(run_dir):
    """Carrega arrays depth/flow e aplica o filtro de sincronizacao."""

    manifesto = carregar_manifesto_memmap(run_dir)
    n = int(manifesto.get("num_samples", 0))
    intervalos_mm = abrir_array_memmap(run_dir, manifesto, "intervals")
    intervalos_brutos = pd.DataFrame.from_records(intervalos_mm[:n]).copy()
    indices_validos = indices_intervalos_sincronizados(intervalos_brutos)
    intervalos = intervalos_brutos.iloc[indices_validos].reset_index(drop=True)
    if not intervalos.empty:
        intervalos["run_id"] = run_dir.name
        intervalos["ordem_intervalo"] = np.arange(len(intervalos))
    return {
        "run_dir": run_dir,
        "manifesto": manifesto,
        "intervalos": intervalos,
        "image_delta_bgr": abrir_array_memmap(run_dir, manifesto, "image_delta_bgr")[indices_validos],
        "depth_delta_log": abrir_array_memmap(run_dir, manifesto, "depth_delta_log")[indices_validos],
        "depth_delta_mask": abrir_array_memmap(run_dir, manifesto, "depth_delta_mask")[indices_validos],
        "flow_vectors": abrir_array_memmap(run_dir, manifesto, "flow_vectors")[indices_validos],
    }


def carregar_todas_runs_depth_memmap(base_dir):
    """Carrega todas as runs depth validas, isolando falhas por run."""

    runs = []
    for run_dir in listar_runs_depth_memmap(base_dir):
        try:
            run = carregar_run_depth_memmap(run_dir)
        except Exception as exc:
            print(f"Run ignorada em {run_dir.name}: {exc}")
            continue
        if len(run["intervalos"]) > 0:
            runs.append(run)
    return runs

In [7]:
raiz_projeto = localizar_raiz_projeto_memmap()
runs_depth_memmap = carregar_todas_runs_depth_memmap(raiz_projeto)

if not runs_depth_memmap:
    display(Markdown(
        "Nenhuma run nova em memmap encontrada em `datasets/depth_ground_truth/run_*/manifest.json`. "
        "Colete uma nova run com `save_ground_truth_dataset:=true`."
    ))
else:
    depth_run = runs_depth_memmap[-1]
    depth_df = depth_run["intervalos"].copy()
    manifesto = depth_run["manifesto"]
    print(f"Run analisada: {depth_run['run_dir'].name}")
    print(f"Intervalos depth/flow validos: {len(depth_df)}")
    print(f"Schema: {manifesto.get('schema_version')}")
    colunas_resumo = [
        "dt_s", "depth_age_s", "depth_dt_s", "delta_x_m", "delta_y_m", "delta_z_m",
        "delta_roll_rad", "delta_pitch_rad", "delta_yaw_heading_rad", "flow_valid_points",
        "flow_track_retention_pct", "flow_mag_p90_px", "radial_flow_p90_px",
        "delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_close_5m_pp",
    ]
    display(depth_df[[c for c in colunas_resumo if c in depth_df.columns]].describe().T)
    fig_depth_delta = go.Figure()
    for coluna, nome in [("delta_depth_p10_m", "Delta depth P10"), ("delta_depth_p50_m", "Delta depth P50"), ("delta_depth_close_5m_pp", "Delta pixels < 5 m")]:
        if coluna in depth_df.columns:
            fig_depth_delta.add_trace(go.Scatter(x=depth_df["ordem_intervalo"], y=depth_df[coluna], mode="lines+markers", name=nome))
    fig_depth_delta.update_layout(title="Variacao de profundidade/proximidade por intervalo visual", xaxis_title="Intervalo visual em ordem de coleta", yaxis_title="Delta do intervalo", template="plotly_white", hovermode="x unified")
    fig_depth_delta.show()
    fig_flow_depth = px.scatter(depth_df, x="flow_mag_p90_px", y="delta_depth_close_5m_pp", color="radial_flow_p90_px", size="flow_valid_points", hover_data=["sample_id", "dt_s", "delta_x_m", "delta_yaw_heading_rad"], title="Flow visual x variacao de ocupacao proxima", template="plotly_white")
    fig_flow_depth.show()
    ranking = depth_df.assign(impacto_proximidade=depth_df["delta_depth_close_5m_pp"].abs()).sort_values(["impacto_proximidade", "flow_mag_p90_px"], ascending=False)
    display(Markdown("**Intervalos mais informativos para inspecao/treino:**"))
    display(ranking[["run_id", "sample_id", "dt_s", "flow_valid_points", "flow_mag_p90_px", "radial_flow_p90_px", "delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_close_5m_pp", "delta_x_m", "delta_y_m", "delta_yaw_heading_rad"]].head(12))

Run analisada: run_20260723_140857
Intervalos depth/flow validos: 39
Schema: depth_interval_memmap_v1


,count,mean,std,min,25%,50%,75%,max
dt_s,39.0,0.085538,0.030815,0.032000,0.064000,0.068000,0.100000,0.164000
depth_age_s,39.0,0.046256,0.016359,0.032000,0.032000,0.036000,0.064000,0.068000
depth_dt_s,39.0,0.093538,0.048814,0.032000,0.064000,0.068000,0.116000,0.196000
delta_x_m,39.0,0.023683,0.292373,-0.692825,-0.208733,0.000431,0.180447,0.611781
delta_y_m,39.0,-0.035597,0.512280,-1.487534,-0.412875,0.000603,0.340623,0.965191
delta_z_m,39.0,-0.001719,0.007794,-0.027992,-0.003899,-0.000450,0.001118,0.021042
delta_roll_rad,39.0,-0.012299,0.066197,-0.340371,-0.017818,0.000000,0.012902,0.113126
delta_pitch_rad,39.0,0.007639,0.056275,-0.161336,-0.000049,0.000054,0.013899,0.267714
delta_yaw_heading_rad,39.0,-0.007906,0.033101,-0.107480,-0.010201,-0.000068,0.004361,0.084217
flow_valid_points,39.0,83.589744,9.599117,63.000000,77.000000,82.000000,90.000000,104.000000


**Intervalos mais informativos para inspecao/treino:**

,run_id,sample_id,dt_s,flow_valid_points,flow_mag_p90_px,radial_flow_p90_px,delta_depth_p10_m,delta_depth_p50_m,delta_depth_close_5m_pp,delta_x_m,delta_y_m,delta_yaw_heading_rad
22,run_20260723_140857,25,0.100,75,24.504078,20.477148,3.234124,-0.128451,-11.541803,0.058506,0.583145,-0.011917
13,run_20260723_140857,15,0.068,86,10.736123,7.405002,0.079784,1.025337,-4.871015,-0.225128,0.253355,-0.006362
17,run_20260723_140857,19,0.064,82,6.212268,6.045126,0.003102,0.262347,-4.515744,-0.261719,0.375919,-0.003308
11,run_20260723_140857,13,0.164,80,39.177437,39.162476,-0.115749,0.593661,-4.003391,-0.352810,0.648306,0.023454
28,run_20260723_140857,31,0.164,66,41.283489,31.883463,0.064251,0.092004,-4.002716,0.458481,-1.487534,-0.016178
32,run_20260723_140857,35,0.132,76,119.970238,72.483093,-0.074771,0.114327,-3.950818,0.544235,-1.031425,-0.033265
24,run_20260723_140857,27,0.132,79,228.804184,212.387085,-0.217927,-0.868989,3.357228,0.149475,0.124847,-0.105597
16,run_20260723_140857,18,0.068,90,10.437965,8.296592,-0.253455,-0.361564,2.761701,-0.308926,0.332989,-0.003633
38,run_20260723_140857,42,0.096,79,46.449455,38.207253,0.124405,0.290844,-2.382996,0.611781,-0.527064,0.020824
25,run_20260723_140857,28,0.064,63,12.754393,12.255019,-0.160876,-0.206499,2.248679,0.168190,0.059387,-0.084870


### Baseline MLP com vetores de variacao

A MLP agora trabalha com features tabulares de deslocamento entre intervalos visuais: deltas de estado, IMU, comandos, estatisticas do optical flow e estatisticas da diferenca de imagem estabilizada. Os alvos tambem sao deltas de depth/proximidade, e nao profundidades absolutas.

A comparacao contra `DummyRegressor(strategy="mean")` continua sendo usada como referencia minima: a MLP so e util se aprender uma relacao melhor que prever a variacao media observada no treino.

Para medir o efeito do aumento de dados sem alterar a dificuldade da avaliacao, as comparacoes cumulativas usam desde o primeiro marco as mesmas runs de validacao e teste. Apenas as runs de treino crescem entre os marcos. A validacao fixa e usada para decisoes de modelagem; o teste fixo serve somente para a comparacao final da curva de aprendizado.

In [8]:
TARGETS_MLP_DELTA = ["delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_p90_m", "delta_depth_close_2m_pp", "delta_depth_close_5m_pp", "delta_depth_close_10m_pp"]


def features_image_delta(img):
    arr = np.asarray(img, dtype=np.float32)
    abs_arr = np.abs(arr)
    return {"img_delta_mean": float(arr.mean()), "img_delta_std": float(arr.std()), "img_delta_abs_mean": float(abs_arr.mean()), "img_delta_abs_p90": float(np.percentile(abs_arr, 90))}


def features_flow_vectors(vetores, valid_points):
    n = int(max(0, min(valid_points, len(vetores))))
    if n == 0:
        return {"flow_vec_rel_std": 0.0, "flow_vec_xy_mean": 0.0, "flow_vec_radial_mean": 0.0, "flow_vec_risk_mean": 0.0}
    v = np.asarray(vetores[:n], dtype=np.float32)
    return {"flow_vec_rel_std": float(v[:, :2].std()), "flow_vec_xy_mean": float(v[:, 2:4].mean()), "flow_vec_radial_mean": float(v[:, 4].mean()), "flow_vec_risk_mean": float(v[:, 5].mean())}


def montar_dataset_mlp_delta_depth(base_dir):
    """Monta features, alvos e metadados dos intervalos depth/flow validos."""

    linhas_x, linhas_y, linhas_meta = [], [], []
    for run in carregar_todas_runs_depth_memmap(base_dir):
        df = run["intervalos"].reset_index(drop=True)
        for i, row in df.iterrows():
            if any(col not in row.index or not np.isfinite(row[col]) for col in TARGETS_MLP_DELTA):
                continue
            feats = {}
            for coluna in df.select_dtypes(include=[np.number]).columns:
                if coluna in TARGETS_MLP_DELTA or coluna.startswith("delta_depth_") or coluna in {"delta_valid_px_pct", "sample_id", "ordem_intervalo"}:
                    continue
                valor = row[coluna]
                feats[coluna] = 0.0 if pd.isna(valor) else float(valor)
            feats.update(features_image_delta(run["image_delta_bgr"][i]))
            feats.update(features_flow_vectors(run["flow_vectors"][i], row.get("flow_valid_points", 0)))
            linhas_x.append(feats)
            linhas_y.append({alvo: float(row[alvo]) for alvo in TARGETS_MLP_DELTA})
            linhas_meta.append({"run_id": run["run_dir"].name, "sample_id": int(row.get("sample_id", i + 1)), "ordem_intervalo": int(row.get("ordem_intervalo", i))})
    return pd.DataFrame(linhas_x), pd.DataFrame(linhas_y), pd.DataFrame(linhas_meta)


RUNS_VALIDACAO_FIXAS = {"run_20260715_161701", "run_20260715_161906", "run_20260715_220346"}
RUNS_TESTE_FIXAS = {"run_20260715_163825","run_20260716_221426"}
MARCOS_BASE_COMPARACAO = (10, 20, 30)


def marcos_comparacao(total_runs):
    """Retorna os marcos historicos e inclui sempre o total atual de runs."""

    marcos = [marco for marco in MARCOS_BASE_COMPARACAO if marco <= total_runs]
    if total_runs and total_runs not in marcos:
        marcos.append(total_runs)
    return tuple(marcos)


def recortar_marco_runs(X, y, meta_df, runs_ordenadas, marco_runs):
    """Recorta X, y e metadados para um marco cumulativo de runs."""

    runs_marco = set(runs_ordenadas[:marco_runs])
    mascara = meta_df["run_id"].isin(runs_marco).to_numpy()
    return tuple(df.loc[mascara].reset_index(drop=True) for df in (X, y, meta_df))


def separar_intervalos(meta_df):
    """Separa intervalos por runs fixas, com fallback temporal para datasets antigos."""

    n = len(meta_df)
    indices = np.arange(n)
    runs = set(meta_df["run_id"].dropna().unique()) if n else set()
    runs_fixas = RUNS_VALIDACAO_FIXAS | RUNS_TESTE_FIXAS
    if runs_fixas.issubset(runs):
        train_runs = runs - runs_fixas
        if not train_runs:
            raise ValueError("Nenhuma run restante para treino apos aplicar os splits fixos.")
        return {
            "train": indices[meta_df["run_id"].isin(train_runs).to_numpy()],
            "val": indices[meta_df["run_id"].isin(RUNS_VALIDACAO_FIXAS).to_numpy()],
            "test": indices[meta_df["run_id"].isin(RUNS_TESTE_FIXAS).to_numpy()],
            "modo": "por run_id com validacao/teste fixos",
            "train_runs": sorted(train_runs),
            "val_runs": sorted(RUNS_VALIDACAO_FIXAS),
            "test_runs": sorted(RUNS_TESTE_FIXAS),
        }
    n_train = max(1, int(n * 0.70))
    n_val = max(1, int(n * 0.15))
    return {
        "train": indices[:n_train],
        "val": indices[n_train:n_train + n_val],
        "test": indices[n_train + n_val:],
        "modo": "temporal por intervalos",
    }


def avaliar_delta(y_true, y_pred, targets, modelo, split):
    """Calcula MAE, RMSE e correlacao de cada alvo de profundidade."""

    linhas = []
    for i, alvo in enumerate(targets):
        real = np.asarray(y_true[:, i], dtype=float)
        pred = np.asarray(y_pred[:, i], dtype=float)
        corr = float(np.corrcoef(real, pred)[0, 1]) if len(real) > 1 and np.std(real) > 1e-9 and np.std(pred) > 1e-9 else np.nan
        linhas.append({"modelo": modelo, "split": split, "alvo_delta": alvo, "MAE": float(mean_absolute_error(real, pred)), "RMSE": float(mean_squared_error(real, pred) ** 0.5), "corr": corr})
    return linhas


def treinar_avaliar_mlp_delta_depth(X, y, meta_df):
    """Treina a MLP multialvo e o baseline nos mesmos splits."""

    targets = list(y.columns)
    split = separar_intervalos(meta_df)
    if len(split["test"]) == 0:
        split["test"] = split["val"]
    if len(split["val"]) == 0:
        split["val"] = split["test"]
    Xv = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    yv = y.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    mlp = TransformedTargetRegressor(regressor=Pipeline([("x_scaler", StandardScaler()), ("mlp", MLPRegressor(hidden_layer_sizes=(64, 16), activation="relu", solver="lbfgs", alpha=0.01, max_iter=2000, random_state=42))]), transformer=StandardScaler())
    dummy = DummyRegressor(strategy="mean")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mlp.fit(Xv[split["train"]], yv[split["train"]])
    dummy.fit(Xv[split["train"]], yv[split["train"]])
    linhas, predicoes = [], {}
    for nome_split, idx in [("treino", split["train"]), ("validacao", split["val"]), ("teste", split["test"])]:
        pred_mlp = mlp.predict(Xv[idx])
        pred_dummy = dummy.predict(Xv[idx])
        predicoes[nome_split] = {"idx": idx, "real": yv[idx], "mlp": pred_mlp, "dummy": pred_dummy}
        linhas.extend(avaliar_delta(yv[idx], pred_mlp, targets, "MLP", nome_split))
        linhas.extend(avaliar_delta(yv[idx], pred_dummy, targets, "Media treino", nome_split))
    return mlp, dummy, split, pd.DataFrame(linhas), predicoes


def calcular_curvas_convergencia_mlp(X, y, meta_df, epocas=200):
    """Calcula MSE padronizado de treino, validacao e teste a cada epoca."""

    split = separar_intervalos(meta_df)
    Xv = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    yv = y.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    x_scaler = StandardScaler().fit(Xv[split["train"]])
    y_scaler = StandardScaler().fit(yv[split["train"]])
    X_scaled = x_scaler.transform(Xv)
    y_scaled = y_scaler.transform(yv)

    modelo = MLPRegressor(
        hidden_layer_sizes=(64, 16),
        activation="relu",
        solver="adam",
        alpha=0.01,
        batch_size=min(64, len(split["train"])),
        learning_rate_init=0.001,
        max_iter=1,
        random_state=42,
    )
    historico = []
    indices_por_split = {
        "treino": split["train"],
        "validacao": split["val"],
        "teste": split["test"],
    }
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for epoca in range(1, epocas + 1):
            modelo.partial_fit(X_scaled[split["train"]], y_scaled[split["train"]])
            for nome_split, idx in indices_por_split.items():
                pred = modelo.predict(X_scaled[idx])
                historico.append({
                    "epoca": epoca,
                    "split": nome_split,
                    "mse_padronizado": float(mean_squared_error(y_scaled[idx], pred)),
                })
    return modelo, pd.DataFrame(historico)


raiz_mlp = localizar_raiz_projeto_memmap()
X_mlp, y_mlp, meta_mlp = montar_dataset_mlp_delta_depth(raiz_mlp)
if len(X_mlp) < 12:
    display(Markdown("Dataset insuficiente para treinar a MLP de deltas. Colete mais intervalos depth/flow em memmap."))
else:
    runs_ordenadas_mlp = list(meta_mlp["run_id"].drop_duplicates())
    resultados_por_volume = []
    artefatos_por_volume = {}
    for marco_runs in marcos_comparacao(len(runs_ordenadas_mlp)):
        if len(runs_ordenadas_mlp) < marco_runs:
            continue
        X_marco, y_marco, meta_marco = recortar_marco_runs(
            X_mlp, y_mlp, meta_mlp, runs_ordenadas_mlp, marco_runs
        )
        modelo, baseline, split, resultados, predicoes = treinar_avaliar_mlp_delta_depth(X_marco, y_marco, meta_marco)
        resultados = resultados.assign(
            marco_runs=marco_runs,
            estrategia_split="validacao_teste_fixos",
        )
        resultados_por_volume.append(resultados)
        artefatos_por_volume[marco_runs] = {"modelo": modelo, "baseline": baseline, "split": split, "predicoes": predicoes, "meta": meta_marco}

    resultados_mlp_cumulativos = pd.concat(resultados_por_volume, ignore_index=True)
    caminho_resultados_mlp = raiz_mlp / "estudos_e_analises" / "comparacao_mlp_splits_fixos.csv"
    resultados_mlp_cumulativos.to_csv(caminho_resultados_mlp, index=False)
    marco_atual = max(artefatos_por_volume)
    artefato_atual = artefatos_por_volume[marco_atual]
    modelo_mlp_delta = artefato_atual["modelo"]
    baseline_media_delta = artefato_atual["baseline"]
    split_mlp = artefato_atual["split"]
    predicoes_mlp_delta = artefato_atual["predicoes"]
    resultados_mlp_delta = resultados_mlp_cumulativos[resultados_mlp_cumulativos["marco_runs"] == marco_atual].drop(columns="marco_runs")

    display(Markdown(
        f"**Comparacao MLP com splits fixos:** validacao={sorted(RUNS_VALIDACAO_FIXAS)}; "
        f"teste={sorted(RUNS_TESTE_FIXAS)}. Apenas o treino cresce entre "
        f"{list(artefatos_por_volume)} runs."
    ))
    resumo_splits = pd.DataFrame([
        {
            "marco_runs": marco,
            "intervalos": len(artefatos_por_volume[marco]["meta"]),
            "treino": len(artefatos_por_volume[marco]["split"]["train"]),
            "validacao": len(artefatos_por_volume[marco]["split"]["val"]),
            "teste": len(artefatos_por_volume[marco]["split"]["test"]),
            "treino_pct": len(artefatos_por_volume[marco]["split"]["train"]) / len(artefatos_por_volume[marco]["meta"]) * 100.0,
            "validacao_pct": len(artefatos_por_volume[marco]["split"]["val"]) / len(artefatos_por_volume[marco]["meta"]) * 100.0,
            "teste_pct": len(artefatos_por_volume[marco]["split"]["test"]) / len(artefatos_por_volume[marco]["meta"]) * 100.0,
        }
        for marco in artefatos_por_volume
    ])
    display(resumo_splits)
    resultados_teste_cumulativos = resultados_mlp_cumulativos[resultados_mlp_cumulativos["split"] == "teste"]
    display(resultados_teste_cumulativos.sort_values(["alvo_delta", "marco_runs", "modelo"]).reset_index(drop=True))
    px.line(resultados_teste_cumulativos, x="marco_runs", y="MAE", color="modelo", facet_col="alvo_delta", facet_col_wrap=3, markers=True, title=f"Curva de aprendizado no teste fixo: {list(artefatos_por_volume)} runs", template="plotly_white").show()

    loss_por_volume = resultados_mlp_cumulativos.query("modelo == 'MLP'").copy()
    loss_por_volume["MSE"] = loss_por_volume["RMSE"] ** 2
    px.line(
        loss_por_volume,
        x="marco_runs",
        y="MSE",
        color="split",
        facet_col="alvo_delta",
        facet_col_wrap=3,
        markers=True,
        title="Loss da MLP principal por volume: treino crescente, validacao e teste fixos",
        labels={"marco_runs": "Runs disponiveis", "MSE": "MSE", "split": "Conjunto"},
        template="plotly_white",
    ).show()

    X_atual, y_atual, meta_atual = recortar_marco_runs(
        X_mlp, y_mlp, meta_mlp, runs_ordenadas_mlp, marco_atual
    )
    _, curvas_convergencia = calcular_curvas_convergencia_mlp(X_atual, y_atual, meta_atual)
    caminho_curvas_loss = raiz_mlp / "estudos_e_analises" / "curvas_loss_mlp.csv"
    curvas_convergencia.assign(marco_runs=marco_atual).to_csv(caminho_curvas_loss, index=False)
    validacao_loss = curvas_convergencia.query("split == 'validacao'")
    melhor_linha = validacao_loss.loc[validacao_loss["mse_padronizado"].idxmin()]
    melhor_epoca = int(melhor_linha["epoca"])
    fig_loss_epocas = px.line(
        curvas_convergencia,
        x="epoca",
        y="mse_padronizado",
        color="split",
        title=f"Convergencia da MLP com {marco_atual} runs: treino, validacao e teste",
        labels={"epoca": "Epoca", "mse_padronizado": "MSE padronizado", "split": "Conjunto"},
        template="plotly_white",
    )
    fig_loss_epocas.add_vline(
        x=melhor_epoca,
        line_dash="dash",
        annotation_text=f"melhor validacao: epoca {melhor_epoca}",
    )
    fig_loss_epocas.show()

    loss_final = curvas_convergencia.groupby("split", as_index=False).tail(1)
    display(pd.DataFrame([{
        "marco_runs": marco_atual,
        "melhor_epoca_validacao": melhor_epoca,
        "menor_loss_validacao": float(melhor_linha["mse_padronizado"]),
        "loss_treino_final": float(loss_final.query("split == 'treino'")["mse_padronizado"].iloc[0]),
        "loss_validacao_final": float(loss_final.query("split == 'validacao'")["mse_padronizado"].iloc[0]),
        "loss_teste_final": float(loss_final.query("split == 'teste'")["mse_padronizado"].iloc[0]),
    }]))
    display(Markdown(
        "A curva por epoca usa a mesma arquitetura com `Adam`, apenas para diagnosticar convergencia. "
        "Os resultados comparativos principais acima continuam usando `LBFGS`. A melhor epoca continua "
        "sendo escolhida somente pela validacao; a curva de teste e exibida apenas como diagnostico final. "
        "Se a loss de treino continuar caindo enquanto a de validacao sobe, "
        "isso indica overfitting."
    ))

    runs_novas = runs_ordenadas_mlp[20:marco_atual]
    runs_novas_treino = [run_id for run_id in runs_novas if run_id in split_mlp["train_runs"]]
    if runs_novas_treino:
        meta_atual_rotulado = meta_atual.copy()
        for grupo, chave_indices in (("train", "train"), ("val", "val"), ("test", "test")):
            meta_atual_rotulado.loc[split_mlp[chave_indices], "split_grupo"] = grupo
        referencia_validacao = (
            resultados_mlp_delta
            .query("modelo == 'MLP' and split == 'validacao'")
            [["alvo_delta", "MAE"]]
            .rename(columns={"MAE": "mae_com_todas"})
        )
        influencia_runs = []
        for run_removida in runs_novas_treino:
            manter = meta_atual_rotulado["run_id"] != run_removida
            _, _, _, resultados_sem_run, _ = treinar_avaliar_mlp_delta_depth(
                X_atual.loc[manter].reset_index(drop=True),
                y_atual.loc[manter].reset_index(drop=True),
                meta_atual_rotulado.loc[manter].reset_index(drop=True),
            )
            validacao_sem_run = (
                resultados_sem_run
                .query("modelo == 'MLP' and split == 'validacao'")
                [["alvo_delta", "MAE"]]
                .rename(columns={"MAE": "mae_sem_run"})
                .merge(referencia_validacao, on="alvo_delta")
            )
            validacao_sem_run["run_removida"] = run_removida
            validacao_sem_run["ganho_ao_remover_pct"] = (
                (validacao_sem_run["mae_com_todas"] - validacao_sem_run["mae_sem_run"])
                / validacao_sem_run["mae_com_todas"].clip(lower=1e-9)
                * 100.0
            )
            influencia_runs.append(validacao_sem_run)

        influencia_runs_df = pd.concat(influencia_runs, ignore_index=True)
        resumo_influencia_runs = (
            influencia_runs_df
            .groupby("run_removida", as_index=False)
            .agg(
                ganho_medio_ao_remover_pct=("ganho_ao_remover_pct", "mean"),
                alvos_que_melhoram=("ganho_ao_remover_pct", lambda valores: int((valores > 0).sum())),
            )
            .sort_values("ganho_medio_ao_remover_pct", ascending=False)
        )
        display(Markdown(
            "**Sensibilidade das novas runs de treino:** cada run foi removida isoladamente e a MLP foi "
            "retreinada. Ganho positivo significa que a validacao melhorou sem aquela run. Como a "
            "validacao ainda e menor que o treino, este ranking indica suspeitas, nao exclusoes automaticas."
        ))
        display(resumo_influencia_runs.round(2))
        px.bar(
            resumo_influencia_runs,
            x="run_removida",
            y="ganho_medio_ao_remover_pct",
            color="alvos_que_melhoram",
            title="Influencia das novas runs de treino na validacao fixa",
            labels={
                "run_removida": "Run removida",
                "ganho_medio_ao_remover_pct": "Melhora media do MAE ao remover (%)",
                "alvos_que_melhoram": "Alvos que melhoram",
            },
            template="plotly_white",
        ).show()

    alvo_plot = "delta_depth_close_5m_pp" if "delta_depth_close_5m_pp" in y_mlp.columns else y_mlp.columns[0]
    alvo_idx = list(y_mlp.columns).index(alvo_plot)
    pred_teste = predicoes_mlp_delta["teste"]
    meta_teste = artefato_atual["meta"].iloc[pred_teste["idx"]].reset_index(drop=True)
    fig_pred = go.Figure()
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["real"][:, alvo_idx], mode="lines+markers", name="real"))
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["mlp"][:, alvo_idx], mode="lines+markers", name="MLP"))
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["dummy"][:, alvo_idx], mode="lines", name="media treino"))
    fig_pred.update_layout(title=f"Predicao no teste fixo para {alvo_plot} com {marco_atual} runs", xaxis_title="Intervalos de teste em ordem temporal", yaxis_title="Delta do alvo", template="plotly_white", hovermode="x unified"); fig_pred.show()
    display(Markdown("**Intervalos do teste fixo para inspecao:**"))
    display(pd.DataFrame({"run_id": meta_teste["run_id"], "sample_id": meta_teste["sample_id"], f"{alvo_plot}_real": pred_teste["real"][:, alvo_idx], f"{alvo_plot}_mlp": pred_teste["mlp"][:, alvo_idx], f"{alvo_plot}_baseline_media": pred_teste["dummy"][:, alvo_idx]}).head(15))

**Comparacao MLP com splits fixos:** validacao=['run_20260715_161701', 'run_20260715_161906', 'run_20260715_220346']; teste=['run_20260715_163825', 'run_20260716_221426']. Apenas o treino cresce entre [10, 20, 30, 40] runs.

,marco_runs,intervalos,treino,validacao,teste,treino_pct,validacao_pct,teste_pct
0,10,369,185,109,75,50.135501,29.539295,20.325203
1,20,786,602,109,75,76.590331,13.867684,9.541985
2,30,1113,929,109,75,83.468104,9.793351,6.738544
3,40,1261,1077,109,75,85.408406,8.643933,5.947661


,modelo,split,alvo_delta,MAE,RMSE,corr,marco_runs,estrategia_split
0,MLP,teste,delta_depth_close_10m_pp,2.844763,4.741618,-0.000839,10,validacao_teste_fixos
1,Media treino,teste,delta_depth_close_10m_pp,0.717460,1.016003,NaN,10,validacao_teste_fixos
2,MLP,teste,delta_depth_close_10m_pp,2.038776,3.333141,0.061953,20,validacao_teste_fixos
3,Media treino,teste,delta_depth_close_10m_pp,0.732677,1.017756,NaN,20,validacao_teste_fixos
4,MLP,teste,delta_depth_close_10m_pp,1.755946,3.180832,0.120955,30,validacao_teste_fixos
5,Media treino,teste,delta_depth_close_10m_pp,0.734162,1.018133,NaN,30,validacao_teste_fixos
6,MLP,teste,delta_depth_close_10m_pp,1.692329,2.294340,0.237872,40,validacao_teste_fixos
7,Media treino,teste,delta_depth_close_10m_pp,0.730670,1.017301,NaN,40,validacao_teste_fixos
8,MLP,teste,delta_depth_close_2m_pp,2.420083,4.917845,0.101843,10,validacao_teste_fixos
9,Media treino,teste,delta_depth_close_2m_pp,1.214932,1.872270,NaN,10,validacao_teste_fixos


,marco_runs,melhor_epoca_validacao,menor_loss_validacao,loss_treino_final,loss_validacao_final,loss_teste_final
0,40,12,0.58976,0.233061,0.824896,0.991298


A curva por epoca usa a mesma arquitetura com `Adam`, apenas para diagnosticar convergencia. Os resultados comparativos principais acima continuam usando `LBFGS`. A melhor epoca continua sendo escolhida somente pela validacao; a curva de teste e exibida apenas como diagnostico final. Se a loss de treino continuar caindo enquanto a de validacao sobe, isso indica overfitting.

**Sensibilidade das novas runs de treino:** cada run foi removida isoladamente e a MLP foi retreinada. Ganho positivo significa que a validacao melhorou sem aquela run. Como a validacao ainda e menor que o treino, este ranking indica suspeitas, nao exclusoes automaticas.

,run_removida,ganho_medio_ao_remover_pct,alvos_que_melhoram
11,run_20260723_123404,7.64,4
14,run_20260723_125240,5.89,4
19,run_20260723_140857,1.92,3
7,run_20260723_115645,0.82,3
5,run_20260723_111458,-8.13,1
17,run_20260723_130416,-9.17,2
13,run_20260723_123848,-12.26,1
6,run_20260723_114842,-12.50,0
0,run_20260721_124311,-14.70,1
1,run_20260721_130115,-18.99,2


**Intervalos do teste fixo para inspecao:**

,run_id,sample_id,delta_depth_close_5m_pp_real,delta_depth_close_5m_pp_mlp,delta_depth_close_5m_pp_baseline_media
0,run_20260715_163825,1,0.000682,-13.709110,-0.177832
1,run_20260715_163825,2,0.000501,-0.153856,-0.177832
2,run_20260715_163825,3,0.000465,-0.230267,-0.177832
3,run_20260715_163825,4,-0.000115,-0.994076,-0.177832
4,run_20260715_163825,5,-0.000724,-0.561974,-0.177832
5,run_20260715_163825,6,0.000036,-0.172689,-0.177832
6,run_20260715_163825,8,0.424986,4.423662,-0.177832
7,run_20260715_163825,9,-1.214283,-15.314536,-0.177832
8,run_20260715_163825,10,-0.793801,-2.598919,-0.177832
9,run_20260715_163825,11,-0.217404,2.401061,-0.177832


### Modelo em duas etapas: evento de proximidade + regressao nos eventos

Este experimento preserva a MLP multi-output anterior e adiciona uma avaliacao alternativa para o alvo `delta_depth_close_5m_pp`. A ideia e separar a pergunta em duas partes: primeiro detectar se houve uma mudanca relevante de proximidade e, so depois, estimar a magnitude/sinal dessa mudanca.

In [9]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, balanced_accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.neural_network import MLPClassifier

EVENT_TARGET_DELTA = "delta_depth_close_5m_pp"
EVENT_THRESHOLD_PP = 0.5
EVENT_PROBA_THRESHOLD = 0.35
EVENT_TEMPORAL_LAGS = (1, 2, 3)


def signed_log1p_array(y, scale=EVENT_THRESHOLD_PP):
    y = np.asarray(y, dtype=float)
    return np.sign(y) * np.log1p(np.abs(y) / max(scale, 1e-6))


def signed_expm1_array(z, scale=EVENT_THRESHOLD_PP):
    z = np.asarray(z, dtype=float)
    return np.sign(z) * np.expm1(np.abs(z)) * max(scale, 1e-6)


def colunas_temporais_evento(X_base):
    """Seleciona as features adequadas para construir contexto temporal."""

    preferidas = [
        "dt_s", "depth_age_s", "depth_dt_s",
        "delta_x_m", "delta_y_m", "delta_z_m",
        "delta_roll_rad", "delta_pitch_rad", "delta_yaw_heading_rad",
        "flow_valid_points", "flow_track_retention_pct",
        "flow_mag_p90_px", "radial_flow_p90_px", "pan_comp_delta_rad",
        "img_delta_abs_mean", "img_delta_abs_p90",
        "flow_vec_radial_mean", "flow_vec_risk_mean", "flow_vec_rel_std",
    ]
    return [col for col in preferidas if col in X_base.columns]


def adicionar_contexto_temporal_evento(X_base, meta_base, lags=EVENT_TEMPORAL_LAGS):
    """Adiciona lags e tendencias sem misturar intervalos entre runs."""

    X_base = X_base.reset_index(drop=True).copy()
    meta_ordem = meta_base.reset_index(drop=True).copy()
    if "ordem_intervalo" not in meta_ordem.columns:
        meta_ordem["ordem_intervalo"] = np.arange(len(meta_ordem))

    colunas = colunas_temporais_evento(X_base)
    if not colunas:
        return X_base

    trabalho = pd.concat([
        meta_ordem[["run_id", "ordem_intervalo"]].reset_index(drop=True),
        X_base[colunas].reset_index(drop=True),
    ], axis=1)
    trabalho["orig_idx"] = np.arange(len(trabalho))
    trabalho = trabalho.sort_values(["run_id", "ordem_intervalo", "orig_idx"])

    novas_partes = [trabalho]
    for lag in lags:
        lagged = trabalho.groupby("run_id")[colunas].shift(lag)
        lagged.columns = [f"{col}_lag{lag}" for col in colunas]
        novas_partes.append(lagged)

    contexto = pd.concat(novas_partes, axis=1)
    for col in colunas:
        lag3 = f"{col}_lag3"
        if lag3 in contexto.columns:
            contexto[f"{col}_trend3"] = contexto[col] - contexto[lag3]

    contexto = contexto.sort_values("orig_idx")
    contexto = contexto.drop(columns=["run_id", "ordem_intervalo", "orig_idx"], errors="ignore")
    contexto = contexto.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([X_base, contexto.drop(columns=colunas, errors="ignore")], axis=1)


def preparar_dataset_evento_proximidade(X_base, y_base, meta_base, target=EVENT_TARGET_DELTA, threshold=EVENT_THRESHOLD_PP):
    """Cria o alvo binario de proximidade e seu contexto temporal."""

    if target not in y_base.columns:
        raise ValueError(f"Alvo {target} nao encontrado em y_base.")

    X_evento = adicionar_contexto_temporal_evento(X_base, meta_base)
    y_delta = y_base[target].reset_index(drop=True).astype(float)
    y_evento = (np.abs(y_delta) >= threshold).astype(int)
    meta_evento = meta_base.reset_index(drop=True).copy()
    meta_evento["delta_target"] = y_delta
    meta_evento["evento_proximidade"] = y_evento
    return X_evento, y_delta, y_evento, meta_evento


def criar_classificador_evento():
    """Cria o pipeline padrao do classificador de proximidade."""

    return Pipeline([
        ("x_scaler", StandardScaler()),
        ("mlp_evento", MLPClassifier(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            solver="lbfgs",
            alpha=0.05,
            max_iter=2000,
            random_state=42,
        )),
    ])


def avaliar_classificacao_evento(y_true, y_pred, y_score, modelo, split):
    """Resume desempenho, matriz de confusao e average precision."""

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    if len(np.unique(y_true)) > 1 and np.std(y_score) > 1e-9:
        avg_precision = float(average_precision_score(y_true, y_score))
    else:
        avg_precision = np.nan

    return {
        "modelo": modelo,
        "split": split,
        "event_rate_real": float(np.mean(y_true)),
        "event_rate_pred": float(np.mean(y_pred)),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_evento": float(precision),
        "recall_evento": float(recall),
        "f1_evento": float(f1),
        "avg_precision": avg_precision,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def avaliar_delta_unico(y_true, y_pred, modelo, split, alvo=EVENT_TARGET_DELTA):
    """Avalia um unico alvo continuo apos o gate de evento."""

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    corr = float(np.corrcoef(y_true, y_pred)[0, 1]) if len(y_true) > 1 and np.std(y_true) > 1e-9 and np.std(y_pred) > 1e-9 else np.nan
    return {
        "modelo": modelo,
        "split": split,
        "alvo_delta": alvo,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "corr": corr,
    }



In [10]:
if "X_mlp" not in globals() or "y_mlp" not in globals() or "meta_mlp" not in globals():
    raiz_mlp = localizar_raiz_projeto_memmap()
    X_mlp, y_mlp, meta_mlp = montar_dataset_mlp_delta_depth(raiz_mlp)

if len(X_mlp) < 12 or EVENT_TARGET_DELTA not in y_mlp.columns:
    display(Markdown("Dataset insuficiente para o modelo em duas etapas de proximidade."))
else:
    runs_ordenadas_evento = list(meta_mlp["run_id"].drop_duplicates())
    resultados_classificacao_cumulativos = []
    for marco_runs in marcos_comparacao(len(runs_ordenadas_evento)):
        if len(runs_ordenadas_evento) < marco_runs:
            continue
        X_marco, y_marco, meta_marco = recortar_marco_runs(
            X_mlp, y_mlp, meta_mlp, runs_ordenadas_evento, marco_runs
        )
        X_evento_marco, _, y_evento_marco, meta_evento_marco = preparar_dataset_evento_proximidade(
            X_marco, y_marco, meta_marco
        )
        split_marco = separar_intervalos(meta_evento_marco)
        Xv_marco = X_evento_marco.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
        yv_marco = y_evento_marco.to_numpy(int)
        clf_marco = criar_classificador_evento()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            clf_marco.fit(Xv_marco[split_marco["train"]], yv_marco[split_marco["train"]])
        for nome_split, idx in [("validacao", split_marco["val"]), ("teste", split_marco["test"])]:
            proba = clf_marco.predict_proba(Xv_marco[idx])[:, 1]
            pred = (proba >= EVENT_PROBA_THRESHOLD).astype(int)
            linha = avaliar_classificacao_evento(yv_marco[idx], pred, proba, "MLP evento", nome_split)
            linha["marco_runs"] = marco_runs
            linha["estrategia_split"] = "validacao_teste_fixos"
            resultados_classificacao_cumulativos.append(linha)

    resultados_classificacao_cumulativos = pd.DataFrame(resultados_classificacao_cumulativos)
    caminho_resultados_evento = raiz_mlp / "estudos_e_analises" / "comparacao_eventos_splits_fixos.csv"
    resultados_classificacao_cumulativos.to_csv(caminho_resultados_evento, index=False)
    display(Markdown("**Curva do classificador com validacao e teste fixos:**"))
    display(resultados_classificacao_cumulativos.sort_values(["split", "marco_runs"]).reset_index(drop=True))
    px.line(resultados_classificacao_cumulativos, x="marco_runs", y="balanced_acc", color="split", markers=True, title=f"Balanced accuracy com splits fixos: {sorted(resultados_classificacao_cumulativos['marco_runs'].unique())} runs", template="plotly_white").show()

    X_evento, y_delta_evento, y_evento, meta_evento = preparar_dataset_evento_proximidade(
        X_mlp, y_mlp, meta_mlp
    )
    split_evento = separar_intervalos(meta_evento)
    if len(split_evento["test"]) == 0:
        split_evento["test"] = split_evento["val"]
    if len(split_evento["val"]) == 0:
        split_evento["val"] = split_evento["test"]

    display(Markdown(
        f"**Dataset evento proximidade:** {len(X_evento)} intervalos, {X_evento.shape[1]} features "
        f"com contexto temporal, alvo `{EVENT_TARGET_DELTA}`, limiar={EVENT_THRESHOLD_PP:.2f} p.p. "
        f"Taxa de evento={float(y_evento.mean()):.1%}. Separacao: {split_evento['modo']}. "
        f"Treino={len(split_evento['train'])}, validacao={len(split_evento['val'])}, teste={len(split_evento['test'])}."
    ))

    resumo_runs_evento = meta_evento.groupby("run_id").agg(
        intervalos=("evento_proximidade", "size"),
        eventos=("evento_proximidade", "sum"),
        taxa_evento=("evento_proximidade", "mean"),
        delta_abs_p90=("delta_target", lambda s: float(np.percentile(np.abs(s), 90))),
    ).reset_index()
    display(resumo_runs_evento)

    Xv_evento = X_evento.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    y_delta_v = y_delta_evento.to_numpy(float)
    y_evento_v = y_evento.to_numpy(int)

    train_idx = split_evento["train"]
    train_event_idx = train_idx[y_evento_v[train_idx] == 1]

    if len(np.unique(y_evento_v[train_idx])) < 2 or len(train_event_idx) < 5:
        display(Markdown(
            "A separacao atual nao tem exemplos suficientes de evento no treino para classificador + regressor. "
            "Diminua `EVENT_THRESHOLD_PP` ou colete mais runs com aproximacao real de obstaculos."
        ))
    else:
        clf_evento = criar_classificador_evento()
        clf_dummy = DummyClassifier(strategy="most_frequent")

        reg_evento = Pipeline([
            ("x_scaler", StandardScaler()),
            ("mlp_delta_evento", MLPRegressor(
                hidden_layer_sizes=(64, 16),
                activation="relu",
                solver="lbfgs",
                alpha=0.05,
                max_iter=2000,
                random_state=42,
            )),
        ])
        reg_dummy_evento = DummyRegressor(strategy="median")

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            clf_evento.fit(Xv_evento[train_idx], y_evento_v[train_idx])
            reg_evento.fit(Xv_evento[train_event_idx], signed_log1p_array(y_delta_v[train_event_idx]))

        clf_dummy.fit(Xv_evento[train_idx], y_evento_v[train_idx])
        reg_dummy_evento.fit(Xv_evento[train_event_idx], y_delta_v[train_event_idx])

        linhas_classificacao = []
        linhas_regressao = []
        predicoes_evento = {}

        for nome_split, idx in [("validacao", split_evento["val"]), ("teste", split_evento["test"] )]:
            if len(idx) == 0:
                continue

            proba_evento = clf_evento.predict_proba(Xv_evento[idx])[:, 1]
            pred_evento = (proba_evento >= EVENT_PROBA_THRESHOLD).astype(int)
            pred_evento_dummy = clf_dummy.predict(Xv_evento[idx]).astype(int)
            score_dummy = np.full(len(idx), float(np.mean(y_evento_v[train_idx])))

            linhas_classificacao.append(avaliar_classificacao_evento(
                y_evento_v[idx], pred_evento, proba_evento, "MLP evento", nome_split
            ))
            linhas_classificacao.append(avaliar_classificacao_evento(
                y_evento_v[idx], pred_evento_dummy, score_dummy, "Classe majoritaria", nome_split
            ))

            pred_delta_evento = signed_expm1_array(reg_evento.predict(Xv_evento[idx]))
            pred_delta_gate = np.where(pred_evento == 1, pred_delta_evento, 0.0)
            pred_delta_evento_real_gate = np.where(y_evento_v[idx] == 1, pred_delta_evento, 0.0)
            pred_delta_dummy_gate = np.where(pred_evento == 1, reg_dummy_evento.predict(Xv_evento[idx]), 0.0)
            pred_delta_zero = np.zeros(len(idx), dtype=float)

            linhas_regressao.extend([
                avaliar_delta_unico(y_delta_v[idx], pred_delta_gate, "Pipeline completo", nome_split),
                avaliar_delta_unico(y_delta_v[idx], pred_delta_evento_real_gate, "Evento real + regressor (diagnostico)", nome_split),
                avaliar_delta_unico(y_delta_v[idx], pred_delta_dummy_gate, "Classificador + mediana", nome_split),
                avaliar_delta_unico(y_delta_v[idx], pred_delta_zero, "Sempre prever zero", nome_split),
            ])

            predicoes_evento[nome_split] = {
                "idx": idx,
                "proba_evento": proba_evento,
                "evento_real": y_evento_v[idx],
                "evento_pred": pred_evento,
                "delta_real": y_delta_v[idx],
                "delta_pred_gate": pred_delta_gate,
                "delta_pred_evento_real_gate": pred_delta_evento_real_gate,
            }

        resultados_classificacao_evento = pd.DataFrame(linhas_classificacao)
        resultados_regressao_evento = pd.DataFrame(linhas_regressao)

        display(Markdown(
            "### Etapa 1 - detectar se houve mudanca relevante de proximidade\n"
            "A classe majoritaria preve evento em todos os intervalos. Por isso ela tem recall 1, "
            "mas balanced accuracy 0,5: encontrar todos os eventos nao significa separar bem as duas classes."
        ))
        display(resultados_classificacao_evento.round(3))

        metricas_classificador = (
            resultados_classificacao_evento
            .query("modelo == 'MLP evento'")
            .melt(
                id_vars=["split"],
                value_vars=["balanced_acc", "precision_evento", "recall_evento", "f1_evento"],
                var_name="metrica",
                value_name="valor",
            )
        )
        metricas_classificador["metrica"] = metricas_classificador["metrica"].map({
            "balanced_acc": "Balanced accuracy",
            "precision_evento": "Precisao",
            "recall_evento": "Recall",
            "f1_evento": "F1",
        })
        fig_metricas_evento = px.bar(
            metricas_classificador,
            x="metrica",
            y="valor",
            color="split",
            barmode="group",
            text="valor",
            range_y=[0, 1.05],
            title="Desempenho do classificador de eventos",
            labels={"metrica": "Metrica", "valor": "Resultado", "split": "Conjunto"},
            template="plotly_white",
        )
        fig_metricas_evento.update_traces(texttemplate="%{text:.3f}", textposition="outside")
        fig_metricas_evento.show()

        resultado_clf_teste = resultados_classificacao_evento.query(
            "modelo == 'MLP evento' and split == 'teste'"
        ).iloc[0]
        matriz_teste = np.array([
            [resultado_clf_teste["tn"], resultado_clf_teste["fp"]],
            [resultado_clf_teste["fn"], resultado_clf_teste["tp"]],
        ], dtype=int)
        fig_confusao = go.Figure(go.Heatmap(
            z=matriz_teste,
            x=["Predito: sem evento", "Predito: evento"],
            y=["Real: sem evento", "Real: evento"],
            text=matriz_teste,
            texttemplate="%{text}",
            colorscale="Blues",
            showscale=False,
        ))
        fig_confusao.update_layout(
            title="Onde o classificador acertou e errou no teste fixo",
            template="plotly_white",
            yaxis_autorange="reversed",
        )
        fig_confusao.show()

        display(Markdown(
            "### Etapa 2 - estimar o tamanho da mudanca\n"
            "`Evento real + regressor (diagnostico)` informa ao regressor quais intervalos realmente "
            "tinham evento. Essa opcao nao pode ser usada em producao; ela serve apenas para verificar "
            "se o erro vem principalmente do classificador ou do regressor."
        ))
        resultados_regressao_teste = (
            resultados_regressao_evento
            .query("split == 'teste'")
            .sort_values("MAE")
            .reset_index(drop=True)
        )
        display(resultados_regressao_teste.round(3))
        fig_mae_teste = px.bar(
            resultados_regressao_teste,
            x="modelo",
            y="MAE",
            color="modelo",
            text="MAE",
            title=f"Erro final no teste fixo para {EVENT_TARGET_DELTA}",
            labels={"modelo": "Estrategia", "MAE": "MAE (p.p.)"},
            template="plotly_white",
        )
        fig_mae_teste.update_traces(texttemplate="%{text:.3f}", textposition="outside")
        fig_mae_teste.update_layout(showlegend=False)
        fig_mae_teste.show()

        mae_pipeline = resultados_regressao_teste.loc[
            resultados_regressao_teste["modelo"] == "Pipeline completo", "MAE"
        ].iloc[0]
        mae_evento_real = resultados_regressao_teste.loc[
            resultados_regressao_teste["modelo"] == "Evento real + regressor (diagnostico)", "MAE"
        ].iloc[0]
        variacao_evento_real_pct = (mae_evento_real / max(mae_pipeline, 1e-9) - 1.0) * 100.0
        if abs(variacao_evento_real_pct) <= 5.0:
            leitura_diagnostico = (
                "Os valores ficaram proximos, entao os erros do classificador nao explicam a maior "
                "parte do erro final."
            )
        elif variacao_evento_real_pct < 0.0:
            leitura_diagnostico = (
                "O MAE diminuiu com o evento real, indicando que os erros do classificador contribuem "
                "para o erro final."
            )
        else:
            leitura_diagnostico = (
                "O MAE aumentou mesmo com o evento real. Isso nao indica ganho do classificador; mostra "
                "que o regressor de magnitude ainda esta instavel nos intervalos de evento."
            )
        display(Markdown(
            f"**Leitura direta:** no teste, fornecer o evento real ao regressor mudou o MAE de "
            f"{mae_pipeline:.3f} para {mae_evento_real:.3f} p.p. {leitura_diagnostico}"
        ))

        maior_mae_validacao = resultados_regressao_evento.query("split == 'validacao'")["MAE"].max()
        maior_mae_teste = resultados_regressao_teste["MAE"].max()
        if maior_mae_validacao > max(100.0, maior_mae_teste * 10.0):
            display(Markdown(
                f"**Alerta de instabilidade:** o maior MAE da validacao foi {maior_mae_validacao:.3e} p.p. "
                "Esse valor extremo achatava as demais barras no grafico anterior. O grafico principal acima "
                "mostra somente o teste fixo, mas a divergencia da validacao continua registrada e indica "
                "que o regressor desta segunda etapa ainda nao esta estavel."
            ))

        if "teste" in predicoes_evento:
            pred_teste_evento = predicoes_evento["teste"]
            meta_teste_evento = meta_evento.iloc[pred_teste_evento["idx"]].reset_index(drop=True)
            inspecao_evento = pd.DataFrame({
                "run_id": meta_teste_evento["run_id"],
                "sample_id": meta_teste_evento["sample_id"],
                "delta_real": pred_teste_evento["delta_real"],
                "evento_real": pred_teste_evento["evento_real"],
                "proba_evento": pred_teste_evento["proba_evento"],
                "evento_pred": pred_teste_evento["evento_pred"],
                "delta_pred_gate": pred_teste_evento["delta_pred_gate"],
                "delta_pred_evento_real_gate": pred_teste_evento["delta_pred_evento_real_gate"],
            })
            display(Markdown("**Intervalos de teste mais extremos para inspecao:**"))
            display(inspecao_evento.assign(abs_delta=lambda df: df["delta_real"].abs()).sort_values("abs_delta", ascending=False).drop(columns="abs_delta").head(20))



**Curva do classificador com validacao e teste fixos:**

,modelo,split,event_rate_real,event_rate_pred,balanced_acc,precision_evento,recall_evento,f1_evento,avg_precision,tn,fp,fn,tp,marco_runs,estrategia_split
0,MLP evento,teste,0.733333,0.800000,0.636364,0.800000,0.872727,0.834783,0.814038,8,12,7,48,10,validacao_teste_fixos
1,MLP evento,teste,0.733333,0.760000,0.677273,0.824561,0.854545,0.839286,0.841182,10,10,8,47,20,validacao_teste_fixos
2,MLP evento,teste,0.733333,0.760000,0.745455,0.859649,0.890909,0.875000,0.865136,12,8,6,49,30,validacao_teste_fixos
3,MLP evento,teste,0.733333,0.720000,0.718182,0.851852,0.836364,0.844037,0.846808,12,8,9,46,40,validacao_teste_fixos
4,MLP evento,validacao,0.669725,0.761468,0.529300,0.686747,0.780822,0.730769,0.663035,10,26,16,57,10,validacao_teste_fixos
5,MLP evento,validacao,0.669725,0.752294,0.605403,0.731707,0.821918,0.774194,0.772553,14,22,13,60,20,validacao_teste_fixos
6,MLP evento,validacao,0.669725,0.761468,0.715944,0.795181,0.904110,0.846154,0.790688,19,17,7,66,30,validacao_teste_fixos
7,MLP evento,validacao,0.669725,0.761468,0.653729,0.759036,0.863014,0.807692,0.772842,16,20,10,63,40,validacao_teste_fixos


**Dataset evento proximidade:** 1261 intervalos, 120 features com contexto temporal, alvo `delta_depth_close_5m_pp`, limiar=0.50 p.p. Taxa de evento=65.1%. Separacao: por run_id com validacao/teste fixos. Treino=1077, validacao=109, teste=75.

,run_id,intervalos,eventos,taxa_evento,delta_abs_p90
0,run_20260715_161701,31,25,0.806452,3.917331
1,run_20260715_161906,48,28,0.583333,4.362569
2,run_20260715_162101,38,23,0.605263,2.037627
3,run_20260715_163428,33,22,0.666667,2.758374
4,run_20260715_163825,43,29,0.674419,3.663851
5,run_20260715_215803,46,38,0.826087,3.518113
6,run_20260715_220110,36,24,0.666667,2.562000
7,run_20260715_220346,30,20,0.666667,4.174669
8,run_20260715_220437,32,21,0.656250,3.797189
9,run_20260716_221426,32,26,0.812500,3.435565


### Etapa 1 - detectar se houve mudanca relevante de proximidade
A classe majoritaria preve evento em todos os intervalos. Por isso ela tem recall 1, mas balanced accuracy 0,5: encontrar todos os eventos nao significa separar bem as duas classes.

...

### Etapa 2 - estimar o tamanho da mudanca
`Evento real + regressor (diagnostico)` informa ao regressor quais intervalos realmente tinham evento. Essa opcao nao pode ser usada em producao; ela serve apenas para verificar se o erro vem principalmente do classificador ou do regressor.

,modelo,split,alvo_delta,MAE,RMSE,corr
0,Sempre prever zero,teste,delta_depth_close_5m_pp,1.559,2.108,NaN
1,Classificador + mediana,teste,delta_depth_close_5m_pp,1.720,2.320,-0.153
2,Evento real + regressor (diagnostico),teste,delta_depth_close_5m_pp,1.791,2.675,0.096
3,Pipeline completo,teste,delta_depth_close_5m_pp,2.030,3.090,0.114


**Leitura direta:** no teste, fornecer o evento real ao regressor mudou o MAE de 2.030 para 1.791 p.p. O MAE diminuiu com o evento real, indicando que os erros do classificador contribuem para o erro final.

**Intervalos de teste mais extremos para inspecao:**

,run_id,sample_id,delta_real,evento_real,proba_evento,evento_pred,delta_pred_gate,delta_pred_evento_real_gate
57,run_20260716_221426,15,6.018052,1,0.999909,1,-0.162110,-0.162110
15,run_20260715_163825,18,5.915465,1,0.997852,1,0.961291,0.961291
13,run_20260715_163825,16,-4.624716,1,0.999861,1,-0.026156,-0.026156
59,run_20260716_221426,17,-4.430673,1,0.999701,1,0.327464,0.327464
60,run_20260716_221426,18,3.902039,1,0.999925,1,-0.141591,-0.141591
24,run_20260715_163825,27,-3.887068,1,1.000000,1,-2.509878,-2.509878
17,run_20260715_163825,20,3.785376,1,0.995035,1,-2.774536,-2.774536
33,run_20260715_163825,36,3.693366,1,0.999998,1,-0.674638,-0.674638
36,run_20260715_163825,40,3.545790,1,0.999781,1,-0.880891,-0.880891
28,run_20260715_163825,31,3.489011,1,1.000000,1,0.465109,0.465109


### Validacao da sincronizacao entre flow, IMU e depth

A pergunta principal Ã© se os intervalos salvos em memmap estao coerentes: `dt_s`, `depth_age_s`, `depth_dt_s`, vetores de optical flow e deltas de depth devem descrever a mesma janela temporal da logica de proximidade visual.

In [11]:
COLUNAS_SYNC_INTERVALOS = ["dt_s", "depth_age_s", "depth_dt_s", "flow_valid_points", "flow_track_retention_pct", "flow_mag_p90_px", "radial_flow_p90_px", "delta_depth_close_5m_pp", "delta_x_m", "delta_y_m", "delta_yaw_heading_rad", "pan_comp_delta_rad"]


def montar_dataframe_intervalos_memmap(base_dir):
    """Concatena os intervalos sincronizados de todas as runs depth."""

    runs = carregar_todas_runs_depth_memmap(base_dir)
    if not runs:
        return pd.DataFrame()
    partes = []
    for run in runs:
        df = run["intervalos"].copy()
        df["run_id"] = run["run_dir"].name
        partes.append(df)
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


def resumir_sincronizacao_intervalos(df):
    if df.empty:
        return pd.DataFrame()
    cols = [c for c in COLUNAS_SYNC_INTERVALOS if c in df.columns]
    return df[cols].describe().T


raiz_flow = localizar_raiz_projeto_memmap()
intervalos_flow_df = montar_dataframe_intervalos_memmap(raiz_flow)

if intervalos_flow_df.empty:
    display(Markdown("Nenhum intervalo memmap encontrado para validar sincronizacao flow/depth."))
else:
    print(f"Intervalos avaliados: {len(intervalos_flow_df)}")
    print(intervalos_flow_df.groupby("run_id").size().rename("intervalos"))
    display(resumir_sincronizacao_intervalos(intervalos_flow_df))
    px.box(intervalos_flow_df, x="run_id", y="dt_s", title="Distribuicao do intervalo temporal entre atualizacoes visuais", labels={"dt_s": "Delta de tempo do intervalo visual (s)", "run_id": "Run"}, template="plotly_white").show()
    px.scatter(intervalos_flow_df, x="radial_flow_p90_px", y="delta_depth_close_5m_pp", color="run_id", size="flow_valid_points", hover_data=["sample_id", "dt_s", "depth_age_s", "pan_comp_delta_rad"], title="Flow radial do intervalo x variacao de proximidade no depth", labels={"radial_flow_p90_px": "P90 do flow radial (px)", "delta_depth_close_5m_pp": "Delta pixels < 5 m (p.p.)"}, template="plotly_white").show()
    serie = intervalos_flow_df.reset_index(drop=True)
    fig_tempo = go.Figure()
    fig_tempo.add_trace(go.Scatter(x=serie.index, y=serie["flow_mag_p90_px"], mode="lines", name="P90 flow"))
    fig_tempo.add_trace(go.Scatter(x=serie.index, y=serie["delta_depth_close_5m_pp"], mode="lines", name="Delta pixels < 5m", yaxis="y2"))
    fig_tempo.update_layout(title="Sequencia dos intervalos: flow visual e delta de proximidade", xaxis_title="Intervalos concatenados em ordem de leitura", yaxis=dict(title="P90 flow (px)"), yaxis2=dict(title="Delta pixels < 5m (p.p.)", overlaying="y", side="right"), template="plotly_white", hovermode="x unified")
    fig_tempo.show()

Intervalos avaliados: 1261
run_id
run_20260715_161701    31
run_20260715_161906    48
run_20260715_162101    38
run_20260715_163428    33
run_20260715_163825    43
run_20260715_215803    46
run_20260715_220110    36
run_20260715_220346    30
run_20260715_220437    32
run_20260716_221426    32
run_20260716_221504    34
run_20260716_221545    39
run_20260716_231519    32
run_20260716_231600    29
run_20260721_102934    38
run_20260721_111722    44
run_20260721_112611    50
run_20260721_113351    56
run_20260721_121851    42
run_20260721_123310    53
run_20260721_124311    61
run_20260721_130115    56
run_20260723_104420    70
run_20260723_110817    54
run_20260723_111141    13
run_20260723_111458    17
run_20260723_114842    11
run_20260723_115645    18
run_20260723_122007    13
run_20260723_122050    14
run_20260723_122339     6
run_20260723_123404    16
run_20260723_123607    11
run_20260723_123848    16
run_20260723_125240    17
run_20260723_125508    12
run_20260723_125716     8
run_

,count,mean,std,min,25%,50%,75%,max
dt_s,1261.0,0.107274,0.055596,0.032000,0.068000,0.100000,0.132000,0.464000
depth_age_s,1261.0,0.044600,0.015897,0.032000,0.032000,0.036000,0.064000,0.068000
depth_dt_s,1261.0,0.120546,0.064077,0.032000,0.068000,0.100000,0.164000,0.464000
flow_valid_points,1261.0,81.244251,13.288818,20.000000,76.000000,83.000000,90.000000,134.000000
flow_track_retention_pct,1261.0,97.112846,5.858999,51.111111,96.808510,100.000000,100.000000,100.000000
flow_mag_p90_px,1261.0,37.219692,45.657516,0.000000,6.212268,20.407911,49.968567,316.002319
radial_flow_p90_px,1261.0,31.026550,37.674400,-0.003340,5.177178,17.337473,43.122894,294.929382
delta_depth_close_5m_pp,1261.0,-0.129506,2.946774,-30.047611,-1.184296,-0.000079,0.922265,20.498875
delta_x_m,1235.0,-0.002154,0.444720,-2.187938,-0.243093,-0.000343,0.225867,2.028631
delta_y_m,1235.0,0.005111,0.627261,-3.173023,-0.407784,0.000572,0.396162,3.277081
